In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import warnings
import random
import math


warnings.filterwarnings('ignore')

In [2]:
df_placar2 = pd.read_csv("vlr_placar2.csv")
df_stats2 = pd.read_csv("vlr_stats2.csv")
df_merged = pd.read_csv("merged_status_placar.csv")
df_jog_map = pd.read_csv("jogador_por_mapas.csv")
df_matchups = pd.read_csv("matchups_score.csv")
df_geral = pd.read_csv("df_jogador_geral.csv")


In [26]:
df_jog_map.head(3)

,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
0,infiltrator,KRÜ,Abyss,1.360,293.5,43,30,6,13,87.5,186.0,32.0,5,2,3,2,0,0.0,1.383
1,lukxo,LOUD,Abyss,1.443,279.7,77,44,8,33,75.7,174.7,34.7,12,10,2,3,0,0.0,1.363
2,vora,100T,Abyss,1.385,255.5,33,22,6,11,82.0,163.5,28.5,2,2,0,2,1,0.5,1.304


In [27]:
df_jog_map[df_jog_map["Jogador"] == "lukxo"]

,Jogador,Time,Mapa,R,ACS,K,D,A,+/-,KAST,ADR,HS%,FK,FD,+/-_FK_FD,Jogos,Wins,WinRate,Impacto
1,lukxo,LOUD,Abyss,1.443,279.7,77,44,8,33,75.7,174.7,34.7,12,10,2,3,0,0.00,1.363
94,lukxo,LOUD,Ascent,1.065,221.0,34,27,4,7,59.5,139.5,40.5,9,2,7,2,1,0.50,1.134
187,lukxo,LOUD,Bind,0.903,190.2,57,60,16,-3,67.5,127.2,28.0,7,5,2,4,1,0.25,0.930
266,lukxo,LOUD,Breeze,1.040,181.0,16,16,5,0,75.0,126.0,34.0,2,7,-5,1,1,1.00,0.996
365,lukxo,LOUD,Corrode,0.650,149.0,10,17,4,-7,45.0,104.0,30.0,1,4,-3,1,0,0.00,0.674
442,lukxo,LOUD,Haven,0.979,228.3,104,97,20,7,69.0,149.4,30.7,21,15,6,7,1,0.14,1.139
519,lukxo,LOUD,Icebox,1.153,234.3,59,52,16,7,69.7,154.3,32.3,15,7,8,3,1,0.33,1.204
581,lukxo,LOUD,Lotus,1.413,291.7,77,53,16,24,69.0,188.3,37.0,14,3,11,3,2,0.67,1.355
725,lukxo,LOUD,Pearl,0.895,201.5,29,31,11,-2,57.5,127.5,29.0,1,7,-6,2,0,0.00,0.842
749,lukxo,LOUD,Split,1.220,270.3,68,49,14,19,70.3,171.3,30.3,15,10,5,3,0,0.00,1.266


In [24]:
df_geral[df_geral["Jogador"] == "lukxo"]

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Impacto
4,lukxo,LOUD,1130,950,244,180,204,146,58,1.104,233.8,68.3,152.1,32.6,592,746,32,82,0.773


In [52]:
a = df_jog_map.loc[df_jog_map["Jogador"] == "lukxo", ["Mapa", "Impacto"]].head(1)
b = df_geral.loc[df_geral["Jogador"] == "lukxo", "Impacto"]

a_valor = a["Impacto"].iloc[0]
b_valor = b.iloc[0]

resultado = a_valor / b_valor
resultado


"""
Cacular Impacto_Mapa / Impacto_Geral 
gerar times aleatorios e calcular confrontos
criar target

"""

'\nCacular Impacto_Mapa / Impacto_Geral \ngerar times aleatorios e calcular confrontos\ncriar target\n\n'

In [21]:
df_geral.head(5)

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Impacto
0,Darker,LOUD,262,288,112,-26,34,34,0,0.932,195.5,70.0,123.0,25.0,148,232,4,26,0.593
1,cauanzin,LOUD,1248,1424,512,-176,148,208,-60,0.896,187.7,68.6,124.8,25.1,892,1062,52,110,0.558
2,Virtyy,LOUD,648,694,160,-46,144,182,-38,0.885,195.5,66.2,121.6,24.7,400,532,20,58,0.565
3,pANcada,LOUD,1264,1312,642,-48,132,108,24,0.990,182.5,72.4,124.4,35.9,892,1062,52,110,0.647
4,lukxo,LOUD,1130,950,244,180,204,146,58,1.104,233.8,68.3,152.1,32.6,592,746,32,82,0.773


# Randon Florest 
Aplicação de Randon Florest para descobrir features importantes para a vitoria ou derrota

In [3]:
df = df_merged.copy()
df['Win'] = (df['Time'] == df['Vencedor']).astype(int)
cols_drop = [
    'Mapa',
    'Time',
    'Agente',
    'Vencedor',
    'Adversario',
    'Score_Time',
    'Score_Adv',
    'Mapas_Time',
    'Mapas_Adv',
    'URL',
    'Jogador'
]

df_model = df.drop(columns=cols_drop)

X = df_model.drop(columns=['Win'])
y = df_model['Win']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

model = RandomForestClassifier(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

importances = model.feature_importances_

feat_importance = pd.Series(importances, index=X.columns)
feat_importance = feat_importance.sort_values(ascending=False)

print(feat_importance.head(20))

ACS          0.105667
R            0.104379
ADR          0.096996
HS%          0.095435
KAST         0.093042
+/-          0.085373
A            0.079080
D            0.076351
K            0.071460
+/-_FK_FD    0.056412
FK           0.052348
FD           0.049596
Mapa_Num     0.033860
dtype: float64


# Função de Criação de Times

In [30]:
def gerar_time_aleatorio(df, n=5, seed=None):
    time = df.sample(n=n, random_state=seed).reset_index(drop=True)
    return time

time = gerar_time_aleatorio(df_geral)
time

,Jogador,Time,K,D,A,+/-,FK,FD,+/-_FK_FD,R,ACS,KAST,ADR,HS%,Score_Time,Score_Adv,Mapas_Time,Mapas_Adv,Impacto
0,Okeanos,EG,988,1006,490,-18,130,140,-10,0.963,190.7,72.1,124.4,35.2,748,690,68,52,0.640
1,cortezia,SEN,1796,1812,534,-16,220,218,2,0.936,191.8,68.9,128.9,35.2,1330,1280,118,96,0.645
2,PxS,LEV,340,412,192,-72,48,54,-6,0.870,166.6,71.7,108.0,27.1,288,298,24,24,0.551
3,nataNk,LEV,208,194,112,14,38,42,-4,1.064,207.7,76.1,136.1,25.4,124,172,8,16,0.685
4,Add3r,FUR,56,56,6,0,8,12,-4,0.877,194.7,65.7,125.3,32.7,52,24,6,0,0.604
